<a href="https://colab.research.google.com/github/abay-qkt/line-album-timestamper/blob/main/line_album_timestamper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

手順
1. ローカルPCで、albumsというフォルダの中に、アルバムフォルダを配置します
1. ローカルPCで、albumsをzip圧縮します
1. 左のファイルタブを選択し、ドラッグ＆ドロップで、albums.zipをアップロードします
1. 上メニューから「ランタイム→すべてのセルの実行」
1. 実行が完了すると「timestamped_albums.zip」というフォルダができているのでダウンロードしてください

# exiftoolのインストール

In [ ]:
!apt-get update
!apt-get install -y exiftool

# アップロードしたzipファイルの解凍

In [ ]:
!unzip albums.zip

# 日付情報を入れる

In [ ]:
import re
import subprocess
from pathlib import Path

ROOT = Path("albums")

date_counters = {}

for folder in sorted(ROOT.iterdir()):
    if not folder.is_dir():
        continue

    m = re.match(r"(\d{8})", folder.name)
    if not m:
        print("skip folder:", folder)
        continue

    date = m.group(1)

    yyyy = date[0:4]
    mm = date[4:6]
    dd = date[6:8]

    if date not in date_counters:
        date_counters[date] = 0

    files = []

    for file in list(folder.glob("*.jpg")) + list(folder.glob("*.jpeg")):
        m = re.search(r"-\s*(\d+)", file.name)
        if not m:
            print("skip file:", file)
            continue

        num = int(m.group(1))
        files.append((num, file))

    # 番号降順（古い → 新しい）
    files.sort(reverse=True)

    for num, file in files:

        date_counters[date] += 1
        n = date_counters[date]

        hh = n // 3600
        mi = (n % 3600) // 60
        ss = n % 60

        timestamp = f"{yyyy}:{mm}:{dd} {hh:02}:{mi:02}:{ss:02}"

        subprocess.run([
            "exiftool",
            "-overwrite_original",
            f"-DateTimeOriginal={timestamp}",
            f"-CreateDate={timestamp}",
            f"-ModifyDate={timestamp}",
            str(file)
        ])

        print(file.name, "→", timestamp)

# 日付情報を入れた写真フォルダをzip圧縮

In [ ]:
!zip -r timestamped_albums.zip albums/